# Hands-On 3: Build an Intelligent Router

We use conditional edges to route a query to different nodes based on its category.

#### Imports

We import `TypedDict` for the state schema and `StateGraph`, `START`, `END` from LangGraph to build the graph.

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

#### Define State

We define `QueryState` with fields for the user query, its category, and the final response.

In [2]:
class QueryState(TypedDict):
    user_query: str
    category: str
    response: str

#### Classify Query

The classify node reads the user query and determines its category using simple keyword matching: `technical`, `billing`, or `general`.

In [3]:
def classify_node(state: QueryState):
    text = state["user_query"].lower()
    if "bug" in text or "error" in text or "install" in text:
        category = "technical"
    elif "bill" in text or "refund" in text or "price" in text:
        category = "billing"
    else:
        category = "general"
    print(f"→ Classified as: {category}")
    return {"category": category}

#### Create Specialized Nodes

We create three nodes: `technical_node` for debugging, `billing_node` for invoices/refunds, and `general_node` for other questions.

In [4]:
def technical_node(state: QueryState):
    return {
        "response": f"🛠️ Technical support: Let's debug '{state['user_query']}'. Check logs & reproduce."
    }


def billing_node(state: QueryState):
    return {
        "response": f"💳 Billing support: Regarding '{state['user_query']}', here is your invoice summary."
    }


def general_node(state: QueryState):
    return {
        "response": f"💬 General help: '{state['user_query']}'? Here are our FAQs."
    }

#### Route to Different Nodes

The routing function examines the category and returns the name of the next node to execute.

In [5]:
def route_query(state: QueryState):
    if state["category"] == "technical":
        return "technical_node"
    elif state["category"] == "billing":
        return "billing_node"
    return "general_node"

#### Build Graph

We add the classify node, the three specialized nodes, and connect them using `add_conditional_edges` for the routing logic.

In [6]:
graph = StateGraph(QueryState)

graph.add_node("classify", classify_node)
graph.add_node("technical_node", technical_node)
graph.add_node("billing_node", billing_node)
graph.add_node("general_node", general_node)

graph.add_edge(START, "classify")

graph.add_conditional_edges(
    "classify",
    route_query,
    {
        "technical_node": "technical_node",
        "billing_node": "billing_node",
        "general_node": "general_node"
    }
)

graph.add_edge("technical_node", END)
graph.add_edge("billing_node", END)
graph.add_edge("general_node", END)

router = graph.compile()

#### Test: Technical Query

Invoke the router with a query containing the word "bug" to verify it routes to the technical node.

In [7]:
result = router.invoke({
    "user_query": "I have a bug in my code",
    "category": "",
    "response": ""
})

print("Category:", result["category"])
print(result["response"])

→ Classified as: technical
Category: technical
🛠️ Technical support: Let's debug 'I have a bug in my code'. Check logs & reproduce.


#### Test Billing Query

Invoke with a billing-related query to verify routing to the billing node.

In [8]:
result = router.invoke({
    "user_query": "Show my bill and refund options",
    "category": "",
    "response": ""
})

print("Category:", result["category"])
print(result["response"])

→ Classified as: billing
Category: billing
💳 Billing support: Regarding 'Show my bill and refund options', here is your invoice summary.


#### Test General Query

Invoke with a general question to verify routing to the general node.

In [9]:
result = router.invoke({
    "user_query": "What features do you offer?",
    "category": "",
    "response": ""
})

print("Category:", result["category"])
print(result["response"])

→ Classified as: general
Category: general
💬 General help: 'What features do you offer?'? Here are our FAQs.


#### Summary

Conditional edges enable decision-making workflows. The classify node determines the category, and the router function decides which specialized node to execute.